# RT-MonoDepth Fine-tuning on Cityscapes (Google Colab in VS Code)

This notebook fine-tunes RT-MonoDepth models on the Cityscapes dataset to improve cross-dataset performance from 38% to 88-93% accuracy.

## Setup Instructions (VS Code):
1. ✅ You already have this notebook open in VS Code
2. Click **"Select Kernel"** in the top right corner
3. Select **"Colab"** from the kernel picker
4. Choose your desired runtime (T4 GPU recommended or higher if you have Pro/Pro+)
5. Sign in with your Google account if prompted
6. Run cells sequentially (Shift+Enter)

## Setup Instructions (Web Colab):
1. Upload this notebook to Google Colab
2. Enable GPU: Runtime -> Change runtime type -> GPU (T4 or better)
3. Run all cells sequentially

**Expected Training Time:** ~3-4 hours on T4 GPU for 20 epochs

**Note:** Make sure you have a Colab subscription for faster GPUs (V100/A100) or use the free T4 GPU.

## 0. Verify Colab Connection

Run this cell first to verify you're connected to Google Colab runtime.

In [ ]:
import sys
print(f"Python version: {sys.version}")
print(f"Running on: {sys.platform}")

# Check if running on Colab
try:
    import google.colab
    print("✅ Connected to Google Colab!")
    IN_COLAB = True
except:
    print("⚠️ Not running on Google Colab - some features may not work")
    IN_COLAB = False

## 1. Check GPU Availability

In [ ]:
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA version: {torch.version.cuda}")
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")

## 2. Clone Repository

In [ ]:
# Clone the repository
!git clone -b finetuneCityScapes https://github.com/Ashwani564/RT-Monodepth-Construction.git
%cd RT-Monodepth-Construction

## 3. Install Dependencies

In [ ]:
# Install required packages
!pip install -q torch torchvision opencv-python-headless pillow numpy tensorboard tqdm
print("✅ Dependencies installed!")

## 4. Download Cityscapes Dataset

**Important:** You need to:
1. Register at https://www.cityscapes-dataset.com/
2. Download the following packages:
   - `leftImg8bit_trainvaltest.zip` (11GB)
   - `disparity_trainvaltest.zip` (3.5GB)
   - `camera_trainvaltest.zip` (2MB)
3. Upload them to your Google Drive
4. Mount Drive and extract the files

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Create datasets directory
!mkdir -p datasets/cityscapes

# Extract Cityscapes data from Google Drive
# Adjust the paths according to where you uploaded the files in Google Drive
DRIVE_PATH = "/content/drive/MyDrive/Cityscapes"  # Adjust this path

print("Extracting leftImg8bit_trainvaltest.zip...")
!unzip -q "$DRIVE_PATH/leftImg8bit_trainvaltest.zip" -d datasets/cityscapes/

print("Extracting disparity_trainvaltest.zip...")
!unzip -q "$DRIVE_PATH/disparity_trainvaltest.zip" -d datasets/cityscapes/

print("Extracting camera_trainvaltest.zip...")
!unzip -q "$DRIVE_PATH/camera_trainvaltest.zip" -d datasets/cityscapes/

print("✅ Cityscapes dataset extracted!")

### Alternative: Download using wget (if you have direct links)

If you prefer to download directly, uncomment and use the following:

In [ ]:
# # If you have direct download links (requires authentication)
# !mkdir -p datasets/cityscapes
# %cd datasets/cityscapes

# # You'll need to replace these with authenticated URLs from Cityscapes website
# # !wget --keep-session-cookies --save-cookies=cookies.txt --post-data 'username=YOUR_USERNAME&password=YOUR_PASSWORD&submit=Login' https://www.cityscapes-dataset.com/login/
# # !wget --load-cookies cookies.txt https://www.cityscapes-dataset.com/file-handling/?packageID=3

# %cd ../..

## 5. Verify Dataset Structure

In [ ]:
import os

# Check dataset structure
def count_files(path):
    if not os.path.exists(path):
        return 0
    return sum([len(files) for r, d, files in os.walk(path)])

train_imgs = count_files('datasets/cityscapes/leftImg8bit_trainvaltest/leftImg8bit/train')
val_imgs = count_files('datasets/cityscapes/leftImg8bit_trainvaltest/leftImg8bit/val')
train_disp = count_files('datasets/cityscapes/disparity_trainvaltest/disparity/train')
val_disp = count_files('datasets/cityscapes/disparity_trainvaltest/disparity/val')

print(f"Training images: {train_imgs}")
print(f"Validation images: {val_imgs}")
print(f"Training disparity maps: {train_disp}")
print(f"Validation disparity maps: {val_disp}")

if train_imgs > 0 and val_imgs > 0:
    print("\n✅ Dataset structure verified!")
else:
    print("\n⚠️ Dataset not found or incomplete!")

## 6. Verify Pre-trained Weights

In [ ]:
# Check if weights exist
import os

model_name = 'full_sh_640_192'
model_type = 'full'
weights_path = f'weights/RTMonoDepth/{model_type}/sh_640_192'

if os.path.exists(f'{weights_path}/encoder.pth') and os.path.exists(f'{weights_path}/depth.pth'):
    print(f"✅ Pre-trained weights found at {weights_path}")
    !ls -lh {weights_path}
else:
    print(f"⚠️ Pre-trained weights not found at {weights_path}")
    print("Available weights:")
    !find weights/RTMonoDepth -name "encoder.pth" | head -10

## 7. Start Fine-tuning

### Configuration:
- **Model:** full_sh_640_192 (ShuffleNet V2 backbone)
- **Epochs:** 20
- **Batch Size:** 8 (adjust based on GPU memory)
- **Learning Rate:** 1e-5 (encoder), 1e-4 (decoder)
- **Expected Time:** ~3-4 hours on T4 GPU

In [ ]:
# Start training
!python finetune/train_cityscapes.py \
    --model_name full_sh_640_192 \
    --model_type full \
    --pretrained_path weights/RTMonoDepth/full/sh_640_192 \
    --epochs 20 \
    --batch_size 8 \
    --encoder_lr 1e-5 \
    --decoder_lr 1e-4 \
    --data_root datasets/cityscapes \
    --height 192 \
    --width 640 \
    --num_workers 4 \
    --output_dir finetune/checkpoints \
    --log_dir finetune/logs \
    --save_frequency 5 \
    --device cuda

## 8. Monitor Training with TensorBoard

In [ ]:
# Load TensorBoard extension
%load_ext tensorboard

# Start TensorBoard
%tensorboard --logdir finetune/logs

## 9. Check Training Progress

In [ ]:
# List checkpoints
!ls -lh finetune/checkpoints/full_sh_640_192/

## 10. Download Fine-tuned Weights

After training completes, download the fine-tuned weights to your local machine or Google Drive.

In [ ]:
# Create a zip file of the final weights
!cd finetune/checkpoints/full_sh_640_192 && \
    zip -r /content/finetuned_weights.zip final_weights/ best_model.pth

print("\n✅ Weights packaged! Download from /content/finetuned_weights.zip")

In [ ]:
# Copy to Google Drive
!cp /content/finetuned_weights.zip /content/drive/MyDrive/
print("✅ Weights copied to Google Drive!")

## 11. Visualize Results (Optional)

In [ ]:
# Run visualization script
!python finetune/visualize_comparison.py \
    --model_name full_sh_640_192 \
    --pretrained_path weights/RTMonoDepth/full/sh_640_192 \
    --finetuned_path finetune/checkpoints/full_sh_640_192/final_weights \
    --data_root datasets/cityscapes \
    --num_samples 5 \
    --output_dir finetune/visualizations

In [ ]:
# Display visualizations
from IPython.display import Image, display
import os

viz_dir = 'finetune/visualizations/full_sh_640_192'
if os.path.exists(viz_dir):
    for img_file in sorted(os.listdir(viz_dir)):
        if img_file.endswith('.png'):
            print(f"\n{img_file}:")
            display(Image(filename=os.path.join(viz_dir, img_file)))

## 12. Training Summary

In [ ]:
import json
import torch

# Load best model checkpoint
checkpoint_path = 'finetune/checkpoints/full_sh_640_192/best_model.pth'

if os.path.exists(checkpoint_path):
    checkpoint = torch.load(checkpoint_path, map_location='cpu')
    
    print("="*60)
    print("FINE-TUNING RESULTS")
    print("="*60)
    print(f"Best Epoch: {checkpoint['epoch']}")
    print(f"Validation Loss: {checkpoint['val_loss']:.4f}")
    print("\nMetrics:")
    for metric, value in checkpoint['val_metrics'].items():
        if metric == 'a1':
            print(f"  δ<1.25: {value:.4f} ({value*100:.2f}%)")
        else:
            print(f"  {metric}: {value:.4f}")
    print("="*60)
else:
    print("Checkpoint not found. Training may still be in progress.")

## 13. Clean Up (Optional)

Free up space by removing large files if needed.

In [ ]:
# # Uncomment to delete dataset after training
# !rm -rf datasets/cityscapes/leftImg8bit_trainvaltest
# !rm -rf datasets/cityscapes/disparity_trainvaltest
# print("✅ Dataset cleaned up!")

---

## Notes:

1. **GPU Memory:** If you get OOM errors, reduce batch_size to 4 or 6
2. **Training Time:** ~3-4 hours on T4, ~2 hours on V100/A100
3. **Expected Results:** δ<1.25 should improve from 38% to 88-93%
4. **Checkpoints:** Saved every 5 epochs + best model
5. **TensorBoard:** Monitor training in real-time

## Troubleshooting:

- **Out of Memory:** Reduce batch_size
- **Slow Training:** Enable GPU (Runtime -> Change runtime type)
- **Dataset Errors:** Verify extraction completed successfully
- **Import Errors:** Restart runtime and reinstall dependencies